In [8]:
import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import yfinance as yf

ModuleNotFoundError: No module named 'yfinance'

In [ ]:
import sys

!{sys.executable} -m pip install yfinance

## Pre-Processing

### Data Download & Feature Selection

In [ ]:
FEATURE_COLUMNS = [
    "close",
    "open_price",
    "high",
    "low",
    "volume"
] 

In [ ]:
indonesia_tickers = [
    "BBCA.JK", "BBRI.JK", "BMRI.JK", "BBNI.JK",
    "TLKM.JK", "ASII.JK", "UNTR.JK", "ICBP.JK",
    "INDF.JK", "ANTM.JK", "MDKA.JK", "PGAS.JK",
    "KLBF.JK", "CPIN.JK", "INTP.JK", "SMGR.JK",
    "ITMG.JK", "PTBA.JK", "BRPT.JK", "TPIA.JK",
    "AMRT.JK", "MAPI.JK", "ERAA.JK", "EXCL.JK",
    "ISAT.JK", "GOTO.JK", "JSMR.JK", "SIDO.JK",
    "MEDC.JK", "ADRO.JK"
]

us_tickers = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "JPM", "V", "WMT",
    "COST", "AVGO", "ORCL", "AMD", "INTC",
    "NFLX", "KO", "PEP", "XOM", "CVX",
    "DIS", "BA", "CAT", "CSCO", "CRM",
    "ADBE", "QCOM", "IBM", "MCD", "NKE"
]

# Gabungkan ticker Indonesia dan US
all_tickers = indonesia_tickers + us_tickers

print("Jumlah ticker Indonesia:", len(indonesia_tickers))
print("Jumlah ticker US:", len(us_tickers))
print("Total ticker:", len(all_tickers))

In [ ]:
DATA_DIR = "Data/Data Yahoo"

os.makedirs(DATA_DIR, exist_ok=True)

# Mengambil seluruh data historis yang tersedia
PERIOD = "max"

# Interval data harian
INTERVAL = "1d"

In [ ]:
download_summary = []
failed_tickers = []

for i, ticker in enumerate(all_tickers, start=1):

    print(f"\n[{i}/{len(all_tickers)}] Mengunduh {ticker}...")

    try:

        df = yf.download(
            ticker,
            period=PERIOD,
            interval=INTERVAL,
            auto_adjust=True,
            progress=False,
            threads=False,
            timeout=30
        )

        if df.empty:
            print(f"Data {ticker} kosong.")
            failed_tickers.append(ticker)
            continue

        # Mengatasi MultiIndex pada kolom
        if isinstance(df.columns, pd.MultiIndex):

            if "Close" in df.columns.get_level_values(0):
                df.columns = df.columns.get_level_values(0)

            elif "Close" in df.columns.get_level_values(1):
                df.columns = df.columns.get_level_values(1)

        # Reset index agar tanggal menjadi kolom
        df = df.reset_index()

        # Menyamakan nama kolom
        df = df.rename(columns={
            "Date": "date",
            "Datetime": "date",
            "Open": "open_price",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume"
        })

        # Kolom yang dibutuhkan model
        required_columns = [
            "date",
            "close",
            "open_price",
            "high",
            "low",
            "volume"
        ]

        # Memastikan semua kolom tersedia
        missing_columns = [
            col for col in required_columns
            if col not in df.columns
        ]

        if missing_columns:
            print("Kolom tidak lengkap:", missing_columns)
            failed_tickers.append(ticker)
            continue

        # Memilih kolom yang digunakan
        df = df[required_columns].copy()

        # Konversi tanggal
        df["date"] = pd.to_datetime(df["date"])

        # Menghapus timezone jika ada
        if df["date"].dt.tz is not None:
            df["date"] = df["date"].dt.tz_localize(None)

        # Memastikan fitur numerik
        numeric_columns = [
            "close",
            "open_price",
            "high",
            "low",
            "volume"
        ]

        for col in numeric_columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

        # Menghapus data tidak valid
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna(subset=required_columns)

        # Menghapus duplikasi dan mengurutkan tanggal
        df = df.drop_duplicates(subset=["date"])
        df = df.sort_values("date").reset_index(drop=True)

        # Memastikan harga dan volume valid
        df = df[
            (df["close"] > 0) &
            (df["open_price"] > 0) &
            (df["high"] > 0) &
            (df["low"] > 0) &
            (df["volume"] >= 0)
        ]

        if len(df) == 0:
            print("Tidak ada data valid.")
            failed_tickers.append(ticker)
            continue

        # Membuat nama file
        filename = ticker.replace(".", "_") + ".csv"

        filepath = os.path.join(
            DATA_DIR,
            filename
        )

        # Menyimpan CSV
        df.to_csv(filepath, index=False)

        # Menyimpan informasi dataset
        download_summary.append({
            "Ticker": ticker,
            "Jumlah Data": len(df),
            "Tanggal Awal": df["date"].min().date(),
            "Tanggal Akhir": df["date"].max().date(),
            "File": filename
        })

        print(
            f"Berhasil: {len(df)} data | "
            f"{df['date'].min().date()} - "
            f"{df['date'].max().date()}"
        )

    except Exception as e:

        print(f"Gagal mengunduh {ticker}: {e}")
        failed_tickers.append(ticker)

    # Jeda untuk mengurangi permintaan berturut-turut
    time.sleep(0.5)


# ==========================================
# 4. HASIL DOWNLOAD
# ==========================================

summary_df = pd.DataFrame(download_summary)

if not summary_df.empty:

    summary_df.to_csv(
        os.path.join(DATA_DIR, "download_summary.csv"),
        index=False
    )

    display(summary_df)

    print("\nTotal ticker berhasil:", len(summary_df))
    print("Total observasi:", summary_df["Jumlah Data"].sum())

if failed_tickers:

    failed_df = pd.DataFrame({
        "Ticker Gagal": failed_tickers
    })

    failed_df.to_csv(
        os.path.join(DATA_DIR, "failed_tickers.csv"),
        index=False
    )

    print("\nTicker gagal:", failed_tickers)

print("\nProses download selesai.")

### Min Max Scaler Function Init

In [ ]:
class MinMaxScalerScratch:
    def __init__(self, eps=1e-8):
        self.min_ = None
        self.max_ = None
        self.eps = eps

    def fit(self, X):
        self.min_ = np.min(X, axis=0)
        self.max_ = np.max(X, axis=0)
        return self

    def transform(self, X):
        return (X - self.min_) / (self.max_ - self.min_ + self.eps)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform_target(self, y_norm, target_col_idx=0):
        min_target = self.min_[target_col_idx]
        max_target = self.max_[target_col_idx]
        return y_norm * (max_target - min_target + self.eps) + min_target

### Sliding Window Function Init

In [ ]:
def create_sliding_windows(data, window_size, target_col_idx=0):
    X, y = [], []
    num_rows, num_features = data.shape
    for i in range(window_size, num_rows):
        window = data[i - window_size:i, :]
        X.append(window.flatten())
        y.append(data[i, target_col_idx])
    return np.array(X), np.array(y).reshape(-1, 1)

### Scaler, Sliding Window, Data Split

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

# Define preprocessing parameters
TARGET_INDEX = 0
WINDOW_SIZE = 10

# Initialize dataset containers
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

val_records_by_ticker = {}
test_records_by_ticker = {}

# Define dataset directory
data_dir = r"Data"
stock_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))

# Define input features
FEATURE_COLUMNS = [
    "close",
    "open_price",
    "high",
    "low",
    "volume"
]

for file_path in stock_files:

    # Extract ticker name from filename
    ticker = os.path.basename(file_path).replace(".csv", "")

    # Load stock dataset
    df = pd.read_csv(file_path)

    # Convert date column to datetime format
    df["date"] = pd.to_datetime(df["date"])

    # Sort data chronologically
    df = df.sort_values("date").reset_index(drop=True)

    # Extract numerical features
    features = df[FEATURE_COLUMNS].to_numpy(dtype=float)

    # Skip datasets with insufficient observations
    if len(features) <= WINDOW_SIZE + 20:
        continue

    # Calculate chronological split indices
    n = len(features)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    # Separate training data
    train_raw = features[:train_end]

    # Initialize and fit scaler using training data only
    scaler = MinMaxScalerScratch()
    scaler.fit(train_raw)

    # Transform the entire dataset using training scaler
    all_scaled = scaler.transform(features)

    # Generate sliding windows from the complete dataset
    X_all, y_all = create_sliding_windows(
        all_scaled,
        window_size=WINDOW_SIZE,
        target_col_idx=TARGET_INDEX
    )

    # Split sliding windows based on their target positions
    X_tr = X_all[:train_end - WINDOW_SIZE]
    y_tr = y_all[:train_end - WINDOW_SIZE]

    X_val = X_all[train_end - WINDOW_SIZE:val_end - WINDOW_SIZE]
    y_val = y_all[train_end - WINDOW_SIZE:val_end - WINDOW_SIZE]

    X_te = X_all[val_end - WINDOW_SIZE:]
    y_te = y_all[val_end - WINDOW_SIZE:]

    # Extract validation and testing dates
    dates_val = df["date"].iloc[train_end:val_end].reset_index(drop=True)
    dates_test = df["date"].iloc[val_end:].reset_index(drop=True)

    # Store training datasets
    if len(X_tr) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)

    # Store validation datasets
    if len(X_val) > 0:
        X_val_list.append(X_val)
        y_val_list.append(y_val)

        val_records_by_ticker[ticker] = {
            "X_val": X_val,
            "y_val_norm": y_val,
            "scaler": scaler,
            "dates": dates_val,
            "actual_prices_raw": features[train_end:val_end, TARGET_INDEX]
        }

    # Store testing datasets
    if len(X_te) > 0:
        X_test_list.append(X_te)
        y_test_list.append(y_te)

        test_records_by_ticker[ticker] = {
            "X_test": X_te,
            "y_test_norm": y_te,
            "scaler": scaler,
            "dates": dates_test,
            "actual_prices_raw": features[val_end:, TARGET_INDEX]
        }

# Combine individual stock datasets into global datasets
X_train_global = np.vstack(X_train_list)
y_train_global = np.vstack(y_train_list)

X_val_global = np.vstack(X_val_list)
y_val_global = np.vstack(y_val_list)

X_test_global = np.vstack(X_test_list)
y_test_global = np.vstack(y_test_list)

# Display dataset preparation results
print("=== GLOBAL STOCK DATASET PREPARATION ===")

print(f"Number of Stocks Processed : {len(test_records_by_ticker)}")

print("\nTraining Dataset:")
print(f"X_train: {X_train_global.shape}")
print(f"y_train: {y_train_global.shape}")

print("\nValidation Dataset:")
print(f"X_val: {X_val_global.shape}")
print(f"y_val: {y_val_global.shape}")

print("\nTesting Dataset:")
print(f"X_test: {X_test_global.shape}")
print(f"y_test: {y_test_global.shape}")

## Forecasting MLP Baseline Backpropagation Gradient Descent Vs. MLP with Remora Optimization Algorithm

### Model Config & Architecture

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# =====================================================
# REPRODUCIBILITY
# =====================================================

SEED = 42

# =====================================================
# DATASET
# =====================================================

X_train = np.asarray(X_train_global, dtype=np.float64)
y_train = np.asarray(y_train_global, dtype=np.float64).reshape(-1, 1)

X_val = np.asarray(X_val_global, dtype=np.float64)
y_val = np.asarray(y_val_global, dtype=np.float64).reshape(-1, 1)

X_test = np.asarray(X_test_global, dtype=np.float64)
y_test = np.asarray(y_test_global, dtype=np.float64).reshape(-1, 1)

# Pastikan input berupa matriks 2D
if X_train.ndim != 2:
    raise ValueError("X_train harus memiliki bentuk 2 dimensi.")

if X_val.ndim != 2 or X_test.ndim != 2:
    raise ValueError("X_val dan X_test harus memiliki bentuk 2 dimensi.")

if X_train.shape[1] != 50:
    raise ValueError(
        f"Input seharusnya 150 fitur, tetapi ditemukan {X_train.shape[1]}."
    )

# =====================================================
# MLP ARCHITECTURE
# =====================================================

INPUT_SIZE = X_train.shape[1]
HIDDEN_SIZE = 32
OUTPUT_SIZE = 1

# =====================================================
# BASELINE MLP
# =====================================================

EPOCHS = 200
LEARNING_RATE = 0.001
BATCH_SIZE = 128

# =====================================================
# REMORA OPTIMIZATION ALGORITHM
# =====================================================

POPULATION_SIZE = 150
MAX_ITERATIONS = 200

LOWER_BOUND = -1.0
UPPER_BOUND = 1.0

REMORA_FACTOR = 0.1

# Initial population perturbation
INITIAL_NOISE = 0.1

# =====================================================
# TOTAL PARAMETERS
# =====================================================

TOTAL_PARAMETERS = (
    INPUT_SIZE * HIDDEN_SIZE
    + HIDDEN_SIZE
    + HIDDEN_SIZE * OUTPUT_SIZE
    + OUTPUT_SIZE
)

print("========== EXPERIMENT CONFIGURATION ==========")

print("Training samples :", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Testing samples  :", X_test.shape[0])

print("\nInput neurons    :", INPUT_SIZE)
print("Hidden neurons   :", HIDDEN_SIZE)
print("Output neurons   :", OUTPUT_SIZE)
print("Total parameters :", TOTAL_PARAMETERS)

print("\nBaseline epochs  :", EPOCHS)
print("Learning rate    :", LEARNING_RATE)
print("Batch size       :", BATCH_SIZE)

print("\nROA population   :", POPULATION_SIZE)
print("ROA iterations   :", MAX_ITERATIONS)

### MLP Function from Scratch

In [ ]:
# =====================================================
# ACTIVATION FUNCTION
# =====================================================

def tanh(x):
    return np.tanh(x)


# =====================================================
# FORWARD PROPAGATION
# =====================================================

def forward_propagation(X, weights):

    W1, b1, W2, b2 = weights

    # Input -> Hidden
    Z1 = np.dot(X, W1) + b1
    A1 = tanh(Z1)

    # Hidden -> Output
    Z2 = np.dot(A1, W2) + b2

    # Linear output
    return Z2, A1


# =====================================================
# LOSS FUNCTION
# =====================================================

def calculate_mse(y_true, y_pred):

    return np.mean((y_true - y_pred) ** 2)


def calculate_mae(y_true, y_pred):

    return np.mean(np.abs(y_true - y_pred))


def calculate_rmse(y_true, y_pred):

    return np.sqrt(calculate_mse(y_true, y_pred))


# =====================================================
# WEIGHT INITIALIZATION - XAVIER UNIFORM
# =====================================================

def initialize_weights(
    input_size,
    hidden_size,
    output_size,
    seed=42
):

    rng = np.random.default_rng(seed)

    # Input -> Hidden
    limit1 = np.sqrt(
        6 / (input_size + hidden_size)
    )

    W1 = rng.uniform(
        -limit1,
        limit1,
        size=(input_size, hidden_size)
    )

    b1 = np.zeros((1, hidden_size))

    # Hidden -> Output
    limit2 = np.sqrt(
        6 / (hidden_size + output_size)
    )

    W2 = rng.uniform(
        -limit2,
        limit2,
        size=(hidden_size, output_size)
    )

    b2 = np.zeros((1, output_size))

    return W1, b1, W2, b2


# =====================================================
# CONVERT WEIGHTS TO VECTOR
# =====================================================

def pack_weights(weights):

    W1, b1, W2, b2 = weights

    return np.concatenate([
        W1.flatten(),
        b1.flatten(),
        W2.flatten(),
        b2.flatten()
    ])


# =====================================================
# CONVERT VECTOR TO WEIGHTS
# =====================================================

def unpack_weights(vector):

    idx = 0

    W1 = vector[
        idx:idx + INPUT_SIZE * HIDDEN_SIZE
    ].reshape(INPUT_SIZE, HIDDEN_SIZE)

    idx += INPUT_SIZE * HIDDEN_SIZE

    b1 = vector[
        idx:idx + HIDDEN_SIZE
    ].reshape(1, HIDDEN_SIZE)

    idx += HIDDEN_SIZE

    W2 = vector[
        idx:idx + HIDDEN_SIZE * OUTPUT_SIZE
    ].reshape(HIDDEN_SIZE, OUTPUT_SIZE)

    idx += HIDDEN_SIZE * OUTPUT_SIZE

    b2 = vector[
        idx:idx + OUTPUT_SIZE
    ].reshape(1, OUTPUT_SIZE)

    return W1, b1, W2, b2


# =====================================================
# INITIALIZE SHARED WEIGHTS
# =====================================================

initial_weights = initialize_weights(
    INPUT_SIZE,
    HIDDEN_SIZE,
    OUTPUT_SIZE,
    seed=SEED
)

baseline_initial_weights = tuple(
    w.copy() for w in initial_weights
)

roa_initial_weights = tuple(
    w.copy() for w in initial_weights
)

# Verify identical initialization
assert all(
    np.array_equal(a, b)
    for a, b in zip(
        baseline_initial_weights,
        roa_initial_weights
    )
)

print("Kedua model memiliki bobot awal identik.")

print(
    "Total parameter:",
    len(pack_weights(initial_weights))
)

### Model 1 : MLP with Back Propagation Gradient Descent

In [ ]:
def train_baseline(
    X_train,
    y_train,
    X_val,
    y_val,
    initial_weights
):

    start_time = time.perf_counter()

    # Salin bobot awal
    W1, b1, W2, b2 = tuple(
        w.copy() for w in initial_weights
    )

    history = {
        "train_loss": [],
        "val_loss": []
    }

    rng = np.random.default_rng(SEED)

    n = len(X_train)

    for epoch in range(EPOCHS):

        # Shuffle data training
        indices = rng.permutation(n)

        for start in range(0, n, BATCH_SIZE):

            batch_indices = indices[
                start:start + BATCH_SIZE
            ]

            X_batch = X_train[batch_indices]
            y_batch = y_train[batch_indices]

            # =========================================
            # FORWARD PROPAGATION
            # =========================================

            weights = (W1, b1, W2, b2)

            y_pred, A1 = forward_propagation(
                X_batch,
                weights
            )

            m = len(X_batch)

            # =========================================
            # BACKPROPAGATION
            # =========================================

            # Output layer
            dZ2 = (2.0 / m) * (y_pred - y_batch)

            dW2 = np.dot(A1.T, dZ2)

            db2 = np.sum(
                dZ2,
                axis=0,
                keepdims=True
            )

            # Hidden layer
            dA1 = np.dot(dZ2, W2.T)

            # Derivative of Tanh
            dZ1 = dA1 * (1 - A1 ** 2)

            dW1 = np.dot(X_batch.T, dZ1)

            db1 = np.sum(
                dZ1,
                axis=0,
                keepdims=True
            )

            # =========================================
            # GRADIENT DESCENT UPDATE
            # =========================================

            W1 -= LEARNING_RATE * dW1
            b1 -= LEARNING_RATE * db1

            W2 -= LEARNING_RATE * dW2
            b2 -= LEARNING_RATE * db2

        weights = (W1, b1, W2, b2)

        # =============================================
        # TRAINING LOSS
        # =============================================

        train_pred, _ = forward_propagation(
            X_train,
            weights
        )

        train_loss = calculate_mse(
            y_train,
            train_pred
        )

        history["train_loss"].append(train_loss)

        # Validation dihitung untuk pemantauan saja
        val_pred, _ = forward_propagation(
            X_val,
            weights
        )

        val_loss = calculate_mse(
            y_val,
            val_pred
        )

        history["val_loss"].append(val_loss)

        if (epoch + 1) % 10 == 0:

            print(
                f"Epoch {epoch + 1:3d}/{EPOCHS} | "
                f"Train MSE: {train_loss:.8f} | "
                f"Val MSE: {val_loss:.8f}"
            )
            
    runtime = time.perf_counter() - start_time

    return weights, history,runtime

### Model 2 : MLP with Remora Optimization Algorithm

In [ ]:
# =====================================================
# ROA FITNESS FUNCTION
# =====================================================

def roa_fitness(position, X, y):

    weights = unpack_weights(position)

    prediction, _ = forward_propagation(
        X,
        weights
    )

    return calculate_mse(y, prediction)


# =====================================================
# REMORA OPTIMIZATION ALGORITHM
# =====================================================

def train_roa(
    X_train,
    y_train,
    initial_weights
):
    start_time = time.perf_counter()

    rng = np.random.default_rng(SEED)

    dimension = TOTAL_PARAMETERS

    # Convert shared MLP weights into a vector
    initial_vector = pack_weights(initial_weights)

    # =============================================
    # POPULATION INITIALIZATION
    # =============================================

    population = np.zeros(
        (POPULATION_SIZE, dimension)
    )

    # Individual 1 = exact shared MLP initial weights
    population[0] = initial_vector.copy()

    # Other 19 individuals = perturbations
    for i in range(1, POPULATION_SIZE):

        population[i] = (
            initial_vector
            + rng.normal(
                0,
                INITIAL_NOISE,
                dimension
            )
        )

    population = np.clip(
        population,
        LOWER_BOUND,
        UPPER_BOUND
    )

    # Historical population for experience attack
    previous_population = population.copy()

    # =============================================
    # INITIAL FITNESS
    # =============================================

    fitness = np.array([
        roa_fitness(individual, X_train, y_train)
        for individual in population
    ])

    best_idx = np.argmin(fitness)

    best_position = population[best_idx].copy()

    best_fitness = fitness[best_idx]

    history = []

    print("Initial best fitness:", best_fitness)

    # =============================================
    # MAIN OPTIMIZATION LOOP
    # =============================================

    for iteration in range(MAX_ITERATIONS):

        t = iteration + 1

        # Host feeding range
        V = 2 * (1 - t / MAX_ITERATIONS)

        for i in range(POPULATION_SIZE):

            current = population[i].copy()

            previous = previous_population[i].copy()

            current_fitness = fitness[i]

            # =====================================
            # 1. EXPERIENCE ATTACK
            # =====================================

            rand1 = rng.uniform(0, 1)

            attempted_position = (
                current
                + (current - previous) * rand1
            )

            attempted_position = np.clip(
                attempted_position,
                LOWER_BOUND,
                UPPER_BOUND
            )

            attempted_fitness = roa_fitness(
                attempted_position,
                X_train,
                y_train
            )

            # =====================================
            # 2. SELECT HOST STRATEGY
            # =====================================

            if attempted_fitness < current_fitness:

                # ---------------------------------
                # HOST FEEDING
                # ---------------------------------

                rand2 = rng.uniform(0, 1)

                B = 2 * V * rand2 - V

                candidate = (
                    current
                    + B * (
                        current
                        - REMORA_FACTOR * best_position
                    )
                )

            else:

                # Randomly choose WOA or SFO
                strategy = rng.integers(0, 2)

                if strategy == 0:

                    # -----------------------------
                    # WOA STRATEGY
                    # -----------------------------

                    rand3 = rng.uniform(0, 1)

                    a = -(1 + t / MAX_ITERATIONS)

                    alpha = (
                        rand3 * (a - 1) + 1
                    )

                    D = np.abs(
                        best_position - current
                    )

                    candidate = (
                        D
                        * np.exp(alpha)
                        * np.cos(2 * np.pi * alpha)
                        + current
                    )

                else:

                    # -----------------------------
                    # SFO STRATEGY
                    # -----------------------------

                    random_idx = rng.integers(
                        0,
                        POPULATION_SIZE
                    )

                    random_individual = population[
                        random_idx
                    ].copy()

                    rand4 = rng.uniform(0, 1)

                    candidate = (
                        best_position
                        - (
                            rand4
                            * (
                                best_position
                                + random_individual
                            ) / 2
                            - random_individual
                        )
                    )

            # =====================================
            # 3. BOUNDARY HANDLING
            # =====================================

            candidate = np.clip(
                candidate,
                LOWER_BOUND,
                UPPER_BOUND
            )

            # =====================================
            # 4. FITNESS EVALUATION
            # =====================================

            candidate_fitness = roa_fitness(
                candidate,
                X_train,
                y_train
            )

            # Store previous position
            previous_population[i] = current

            # Update population
            population[i] = candidate

            fitness[i] = candidate_fitness

            # =====================================
            # 5. UPDATE GLOBAL BEST
            # =====================================

            if candidate_fitness < best_fitness:

                best_fitness = candidate_fitness

                best_position = candidate.copy()

        # Record best-so-far fitness
        history.append(best_fitness)

        if (iteration + 1) % 10 == 0:

            print(
                f"Iteration {iteration + 1:3d}/"
                f"{MAX_ITERATIONS} | "
                f"Best MSE: {best_fitness:.8f}"
            )

    runtime = time.perf_counter() - start_time
    
    best_weights = unpack_weights(best_position)

    return best_weights, history, runtime

In [ ]:
baseline_weights, baseline_history, baseline_runtime = train_baseline(
    X_train,
    y_train,
    X_val,
    y_val,
    baseline_initial_weights
)

roa_weights, roa_history, roa_runtime = train_roa(
    X_train,
    y_train,
    roa_initial_weights
)

print(f"Runtime MLP Baseline : {baseline_runtime:.4f} detik")
print(f"Runtime ROA-MLP      : {roa_runtime:.4f} detik")

## Evaluation

In [ ]:
def evaluate_model(X, y, weights):
    """
    Mengevaluasi model menggunakan MSE, MAE, dan RMSE.
    """

    predictions, _ = forward_propagation(X, weights)

    y = np.asarray(y, dtype=np.float64).reshape(-1, 1)
    predictions = np.asarray(predictions, dtype=np.float64).reshape(-1, 1)

    mse = np.mean((y - predictions) ** 2)
    mae = np.mean(np.abs(y - predictions))
    rmse = np.sqrt(mse)

    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse
    }

In [ ]:
# Evaluasi MLP Baseline
baseline_train = evaluate_model(
    X_train, y_train, baseline_weights
)

baseline_val = evaluate_model(
    X_val, y_val, baseline_weights
)

baseline_test = evaluate_model(
    X_test, y_test, baseline_weights
)


# Evaluasi ROA-MLP
roa_train = evaluate_model(
    X_train, y_train, roa_weights
)

roa_val = evaluate_model(
    X_val, y_val, roa_weights
)

roa_test = evaluate_model(
    X_test, y_test, roa_weights
)

In [ ]:
comparison_results = pd.DataFrame([
    {
        "Model": "MLP Baseline",
        "Dataset": "Training",
        **baseline_train
    },
    {
        "Model": "MLP Baseline",
        "Dataset": "Validation",
        **baseline_val
    },
    {
        "Model": "MLP Baseline",
        "Dataset": "Testing",
        **baseline_test
    },
    {
        "Model": "ROA-MLP",
        "Dataset": "Training",
        **roa_train
    },
    {
        "Model": "ROA-MLP",
        "Dataset": "Validation",
        **roa_val
    },
    {
        "Model": "ROA-MLP",
        "Dataset": "Testing",
        **roa_test
    }
])

display(
    comparison_results.style.format({
        "MSE": "{:.8f}",
        "MAE": "{:.8f}",
        "RMSE": "{:.8f}"
    })
)

In [ ]:
def predict_stock(X, weights):
    """
    Menghasilkan prediksi dari model MLP.
    """
    predictions, _ = forward_propagation(X, weights)

    return np.asarray(predictions).reshape(-1, 1)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_stock_predictions(
    test_records_by_ticker,
    baseline_weights,
    roa_weights
):
    for ticker, records in test_records_by_ticker.items():

        # Mengambil data testing masing-masing saham
        X_stock = records["X_test"]
        actual_prices = np.asarray(
            records["actual_prices_raw"]
        ).reshape(-1)

        dates = pd.to_datetime(records["dates"])

        # Prediksi MLP Baseline
        baseline_pred_norm = predict_stock(
            X_stock,
            baseline_weights
        )

        # Prediksi ROA-MLP
        roa_pred_norm = predict_stock(
            X_stock,
            roa_weights
        )

        # Mengembalikan prediksi ke skala harga asli
        baseline_pred = records["scaler"].inverse_transform_target(
            baseline_pred_norm,
            target_col_idx=TARGET_INDEX
        ).reshape(-1)

        roa_pred = records["scaler"].inverse_transform_target(
            roa_pred_norm,
            target_col_idx=TARGET_INDEX
        ).reshape(-1)

        # Membuat grafik
        plt.figure(figsize=(14, 6))

        plt.plot(
            dates,
            actual_prices,
            label="Nilai Riil",
            linewidth=2
        )

        plt.plot(
            dates,
            baseline_pred,
            label="Prediksi MLP Baseline",
            linewidth=1.5,
            linestyle="--"
        )

        plt.plot(
            dates,
            roa_pred,
            label="Prediksi ROA-MLP",
            linewidth=1.5,
            linestyle="-."
        )

        plt.title(
            f"Perbandingan Harga Saham Aktual dan Prediksi - {ticker}",
            fontsize=13
        )

        plt.xlabel("Tanggal")
        plt.ylabel("Harga Saham")
        plt.legend()
        plt.grid(True, alpha=0.3)

        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        print(f"\nTicker: {ticker}")

        print("Jumlah data aktual :", len(actual_prices))
        print("Jumlah prediksi baseline :", len(baseline_pred))
        print("Jumlah prediksi ROA :", len(roa_pred))
        print("Jumlah tanggal :", len(dates))

        print("\nRentang harga aktual:")
        print(np.min(actual_prices), np.max(actual_prices))

        print("\nRentang prediksi baseline:")
        print(np.min(baseline_pred), np.max(baseline_pred))

        print("\nRentang prediksi ROA:")
        print(np.min(roa_pred), np.max(roa_pred))

        print("\nActual normalized:")
        print("Min:", np.min(
            records["y_test_norm"]
        ))
        print("Max:", np.max(
            records["y_test_norm"]
        ))

        print("\nBaseline normalized:")
        print("Min:", np.min(baseline_pred_norm))
        print("Max:", np.max(baseline_pred_norm))

        print("\nROA normalized:")
        print("Min:", np.min(roa_pred_norm))
        print("Max:", np.max(roa_pred_norm))


        

In [ ]:
plot_stock_predictions(
    test_records_by_ticker,
    baseline_weights,
    roa_weights
)

